# 2차시: Multi-Agent (AutoGen) 실습 (Part 1)

AutoGen 기반 멀티 에이전트 시스템 구축 실습 노트북입니다.

## 목차
1. AutoGen 프레임워크 개요
2. 설치 및 기본 설정
3. Agent 구성요소
4. RoundRobinGroupChat (순환 대화)
5. SelectorGroupChat (동적 선택)
6. Swarm (자율 핸드오프)

## 환경 설정

AutoGen 패키지 설치 및 Groq API 키 설정

In [ ]:
# AutoGen 설치 (최초 1회)
!pip install -U "autogen-agentchat" "autogen-ext[openai]"

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

api_key = os.environ.get("GROQ_API_KEY")
print(f"API key loaded: {'Yes' if api_key else 'No'}")

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelFamily

model_client = OpenAIChatCompletionClient(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key,
    model_info={
        "max_tokens": 4096,
        "temperature": 0.7,
        "vision": False,
        "function_calling": True,
        "family": ModelFamily.UNKNOWN,
        "structured_output": True,
        "json_output": True,
    }
)

---
## 3. Agent 구성요소

### 3-1. AssistantAgent 생성 및 실행

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console

agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    system_message="당신은 유용한 AI 비서입니다.",
)

In [ ]:
# 방법 1: 결과 직접 받기
result = await agent.run(task="Python의 장점 3가지를 알려줘")
print(result.messages[-1].content)

In [ ]:
# 방법 2: 스트리밍 출력
await Console(agent.run_stream(task="Python의 장점 3가지를 알려줘"))

### 3-2. UserProxyAgent

사람의 입력을 에이전트 대화에 전달하는 프록시입니다.
- 자신의 차례가 오면 실행을 **중단**하고 사람의 입력을 기다립니다.
- `input_func`에 원하는 입력 함수를 전달합니다.

In [ ]:
from autogen_agentchat.agents import UserProxyAgent

# 실제 사용 시: input_func=input → 콘솔에서 사용자 입력을 기다림
# 노트북에서는 시뮬레이션 함수로 대체
user_proxy = UserProxyAgent(
    "user_proxy",
    input_func=lambda prompt: "Python과 JavaScript 중 어떤 걸 배워야 할까요?",
)

# UserProxyAgent는 자신의 차례에 input_func을 호출하여 사용자 입력을 받음
response = await user_proxy.run(task="프로그래밍 언어에 대해 질문해주세요.")
print(response.messages[-1].content)

### 3-3. 메시지 타입

Agent 간 통신에 사용되는 구조화된 메시지들입니다.

| 메시지 타입 | 내용 | 용도 |
|:---:|---|---|
| `TextMessage` | 문자열 | 일반 텍스트 통신 |
| `MultiModalMessage` | 텍스트 + 이미지 | 멀티모달 입력 |
| `HandoffMessage` | 출발/도착 에이전트 | Swarm 핸드오프 신호 |
| `StopMessage` | 종료 사유 | 대화 종료 신호 |
| `TaskResult` | 메시지 목록 + 종료 사유 | 실행 최종 결과 |

In [ ]:
from autogen_agentchat.messages import (
    TextMessage,
    MultiModalMessage,
    HandoffMessage,
    StopMessage,
)
from autogen_core import Image
from io import BytesIO
import PIL.Image, requests

# 1. TextMessage - 일반 텍스트 통신
text_msg = TextMessage(content="안녕하세요!", source="User")
print("=== TextMessage ===")
print(text_msg)

# 2. MultiModalMessage - 텍스트 + 이미지
print("\n=== MultiModalMessage ===")
pil_img = PIL.Image.open(BytesIO(requests.get("https://picsum.photos/200").content))
img = Image(pil_img)
multi_msg = MultiModalMessage(
    content=["이 이미지를 분석해주세요.", img],
    source="User",
)
print(multi_msg)

# 3. HandoffMessage - Swarm 핸드오프 신호
print("\n=== HandoffMessage ===")
handoff_msg = HandoffMessage(
    source="travel_agent",
    target="refund_agent",
    content="환불 처리를 요청합니다.",
)
print(handoff_msg)

# 4. StopMessage - 대화 종료 신호
print("\n=== StopMessage ===")
stop_msg = StopMessage(content="작업이 완료되었습니다.", source="assistant")
print(stop_msg)

### 3-4. MultiModalMessage 활용: 이미지 추론

이미지를 Agent에게 전달하여 시각적 분석을 수행합니다.
- `content`에 **텍스트와 `Image` 객체를 리스트로** 함께 전달
- 비전 모델(`llama-3.2-90b-vision-preview` 등)이 필요 — 텍스트 전용 모델은 이미지 처리 불가

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import MultiModalMessage
from autogen_agentchat.ui import Console
from autogen_core import Image
from io import BytesIO
import PIL.Image, requests

annotator = AssistantAgent(
    "annotator",
    model_client=model_client,
    system_message="""이미지 분석 전문가입니다.
    이미지를 받으면 다음을 수행하세요:
    1. 이미지에 보이는 주요 객체를 나열
    2. 각 객체의 위치를 설명 (좌상단, 중앙 등)
    3. 전체적인 장면을 한 문장으로 요약""",
)

# URL에서 이미지를 가져와서 Image로 변환
pil_image = PIL.Image.open(BytesIO(requests.get("https://picsum.photos/300/200").content))
img = Image(pil_image)

result = await Console(annotator.run_stream(task=MultiModalMessage(
    source="user",
    content=["이 이미지를 분석해주세요.", img],
)))

---
## 4. RoundRobinGroupChat (순환 대화)

### 4-1. Writer + Critic (Reflection Pattern)

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

writer = AssistantAgent(
    "writer",
    model_client=model_client,
    system_message="당신은 창의적인 작가입니다. 주어진 주제로 글을 작성하세요. 피드백을 받으면 반영하여 수정하세요.",
)
critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="""당신은 글 비평가입니다. 글을 검토하고 구체적인 피드백을 제공하세요.
    첫 번째 리뷰에서는 반드시 개선점을 찾아 피드백을 주세요. 절대 첫 리뷰에서 승인하지 마세요.
    두 번째 리뷰부터 수정이 잘 반영되었으면 'APPROVE'라고 응답하세요.""",
)

# OR 조합: APPROVE 언급 또는 메시지 8회 도달 시 종료 (무한 루프 방지)
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

team = RoundRobinGroupChat([writer, critic], termination_condition=termination)
await Console(team.run_stream(task="가을을 주제로 짧은 시를 써줘"))

---------- TextMessage (user) ----------
가을을 주제로 짧은 시를 써줘
---------- TextMessage (writer) ----------
가을 숲 사이로 바람 노래하고  
단풍잎은 춤추며 땅에 내려앉네  
서늘한 공기 속에 마음도 물들어  
황금 빛 추억 속에 나를 맡기네
---------- TextMessage (critic) ----------
아름다운 가을의 정취를 잘 담아낸 시입니다. 특히 "가을 숲 사이로 바람 노래하고"와 "단풍잎은 춤추며 땅에 내려앉네"라는 구절은 자연의 움직임과 색채를 생생하게 그려내어 독자가 가을 풍경을 머릿속에 쉽게 떠올릴 수 있게 합니다.

시의 흐름도 부드럽고 리듬감이 있어 읽는 이에게 편안함을 줍니다. "서늘한 공기 속에 마음도 물들어"라는 표현은 단순한 자연 묘사를 넘어 마음의 변화를 연결시켜 시에 깊이를 더한 점이 인상적입니다. 마지막 구절 "황금 빛 추억 속에 나를 맡기네"는 가을의 따뜻하고 소중한 느낌을 잘 전달하며 마무리를 아름답게 장식합니다.

한 가지 더 제안하자면, 후렴구나 반복적인 이미지 사용을 통해 시의 중심 메시를 더욱 강조하거나, 감각적인 세부 묘사를 추가하여 독자의 몰입도를 높일 수 있습니다. 예를 들어, 가을바람의 온도나 소리, 단풍잎의 질감 등을 조금 더 구체적으로 표현하면 더욱 풍부한 시가 될 것입니다.

전반적으로 짧지만 가을의 분위기와 감정을 잘 포착한 시로서, 읽는 이로 하여금 가을의 풍경과 정취를 느끼게 하는 훌륭한 작품입니다.
---------- TextMessage (writer) ----------
피드백 감사합니다. 후렴구를 추가하고 감각적인 세부 묘사를 더해 시를 다듬어 보았습니다.

가을 숲 사이로 바람 노래하고  
차가운 숨결에 단풍잎은 춤추네  
부드러운 낙엽 위, 발자국 소리 점점  
가을빛 속에, 마음도 서서히 물들어  

가을 숲 사이로 바람 노래하고  
부스러지는 나뭇잎, 손끝에 닿는 듯  
서늘한 공기 속에 깊은 추억 숨

TaskResult(messages=[TextMessage(id='5e6cdfc0-15ea-4054-8a5e-996676fd8ce2', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 10, 1, 48, 43, 598581, tzinfo=datetime.timezone.utc), content='가을을 주제로 짧은 시를 써줘', type='TextMessage'), TextMessage(id='6e890406-cbae-497f-b736-e8848467797a', source='writer', models_usage=RequestUsage(prompt_tokens=56, completion_tokens=55), metadata={}, created_at=datetime.datetime(2026, 4, 10, 1, 48, 45, 95233, tzinfo=datetime.timezone.utc), content='가을 숲 사이로 바람 노래하고  \n단풍잎은 춤추며 땅에 내려앉네  \n서늘한 공기 속에 마음도 물들어  \n황금 빛 추억 속에 나를 맡기네', type='TextMessage'), TextMessage(id='5a1cef30-6977-4a81-aecc-e27e29f2a003', source='critic', models_usage=RequestUsage(prompt_tokens=110, completion_tokens=339), metadata={}, created_at=datetime.datetime(2026, 4, 10, 1, 48, 49, 428824, tzinfo=datetime.timezone.utc), content='아름다운 가을의 정취를 잘 담아낸 시입니다. 특히 "가을 숲 사이로 바람 노래하고"와 "단풍잎은 춤추며 땅에 내려앉네"라는 구절은 자연의 움직임과 색채를 생생하게 그려내어 독자가 가을 풍경을 머릿속에 쉽게 떠올릴 수 있게 합니다

---
## 5. SelectorGroupChat (동적 선택)

### 5-1. Multi-Expert Collaboration

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

planner = AssistantAgent(
    "PlanningAgent",
    description="작업을 계획하는 에이전트. 새 작업 시 먼저 참여해야 함.",
    model_client=model_client,
    system_message="복잡한 작업을 하위 작업으로 분해하세요.",
)
researcher = AssistantAgent(
    "WebSearchAgent",
    description="웹에서 정보를 검색하는 에이전트.",
    model_client=model_client,
    system_message="검색하여 관련 정보를 반환하세요.",
)
analyst = AssistantAgent(
    "DataAnalystAgent",
    description="데이터 분석 및 계산을 수행하는 에이전트.",
    model_client=model_client,
    system_message="데이터를 분석하고 수치를 계산하세요. 최종 보고가 완료되면 'TERMINATE'라고 응답하세요.",
)

In [ ]:
termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(25)

team = SelectorGroupChat(
    [planner, researcher, analyst],
    model_client=model_client,                # Selector용 LLM
    termination_condition=termination,
    allow_repeated_speaker=True,              # 같은 에이전트 연속 발언 허용
)

await Console(team.run_stream(
    task="테슬라의 최근 주가 동향을 분석해줘"
))

### 5-2. Custom Selector Function

In [ ]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

def custom_selector(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    """항상 Planner를 거치도록 커스텀 라우팅"""
    if messages[-1].source != "PlanningAgent":
        return "PlanningAgent"      # Planner로 돌려보내기
    return None                      # None이면 LLM 기반 선택으로 폴백

team = SelectorGroupChat(
    [planner, researcher, analyst],
    model_client=model_client,
    termination_condition=termination,
    selector_func=custom_selector,   # 커스텀 선택 함수 등록
)

await Console(team.run_stream(
    task="Python과 JavaScript의 장단점을 비교해줘"
))

---
## 6. Swarm (자율 핸드오프)

### 6-1. Customer Service Swarm

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

def refund_flight(flight_id: str) -> str:
    """항공편을 환불 처리합니다."""
    return f"항공편 {flight_id} 환불이 완료되었습니다."

travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""당신은 여행 상담 에이전트입니다. 환불 요청은 flights_refunder에게 위임하세요.
    사용자 정보가 필요하면 user에게 핸드오프하세요. 완료 시 TERMINATE라고 응답하세요.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""당신은 환불 전문 에이전트입니다. refund_flight 도구로 환불을 처리하세요.
    완료 후 travel_agent에게 핸드오프하세요.""",
)

In [ ]:
termination = (
    HandoffTermination(target="user")
    | TextMentionTermination("TERMINATE")
    | MaxMessageTermination(10)
)

team = Swarm(
    [travel_agent, flights_refunder],
    termination_condition=termination,
)

result = await Console(
    team.run_stream(task="항공편 FL-12345의 환불을 요청합니다.")
)

### 6-2. Swarm with User Handoff Loop

In [ ]:
from autogen_agentchat.messages import HandoffMessage

# 대화 초기화
await team.reset()

# === 옵션 A: 시뮬레이션 입력 (노트북용) ===
simulated_inputs = iter(["FL-67890 항공편 환불을 요청합니다.", "감사합니다."])
get_user_input = lambda: next(simulated_inputs)

# === 옵션 B: 실제 사용자 입력 (아래 주석 해제) ===
# get_user_input = lambda: input("사용자: ")

result = await Console(
    team.run_stream(task="항공편 예약 관련 도움이 필요합니다.")
)
last = result.messages[-1]

while isinstance(last, HandoffMessage) and last.target == "user":
    try:
        user_input = get_user_input()
    except (StopIteration, EOFError):
        break
    if user_input.lower() == "exit":
        break
    print(f"사용자: {user_input}")
    result = await Console(team.run_stream(
        task=HandoffMessage(
            source="user", target=last.source, content=user_input
        )
    ))
    last = result.messages[-1]

---
## 7. GraphFlow (그래프 기반 실행 제어)

에이전트 실행 순서를 **DiGraph(방향 그래프)** 로 명시적으로 정의하는 패턴입니다.

| 기존 패턴 (RoundRobin / Selector / Swarm) | GraphFlow |
|:---:|:---|
| 런타임에 다음 발언자 결정 | **그래프 구조로 사전 정의** |
| 유연하지만 비결정론적 | **결정론적, 예측 가능한 흐름** |
| 순차 실행만 가능 | **병렬 실행(Fan-out/Join) 지원** |

지원하는 흐름: **순차(Sequential)** · **병렬(Parallel)** · **조건 분기(Conditional)** · **루프(Loop)**

### 7-1. 콘텐츠 검수 파이프라인 (조건 분기 + 루프)

승인될 때까지 Writer ↔ Reviewer 반복, 승인되면 Publisher로 전달

```
Writer → Reviewer → APPROVE → Publisher (완료)
                  → REJECT  → Writer (루프)
```

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import GraphFlow, DiGraphBuilder
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.ui import Console

writer = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 글을 작성하세요. 피드백이 있으면 반영하여 수정하세요.",
)
reviewer = AssistantAgent(
    "reviewer", model_client=model_client,
    system_message="""글을 검토하세요.
    첫 번째 리뷰에서는 반드시 구체적인 개선점을 제시하세요. 절대 첫 리뷰에서 승인하지 마세요.
    두 번째 리뷰부터 수정이 잘 반영되었으면 반드시 'APPROVE'를 포함하여 응답하세요.""",
)
publisher = AssistantAgent(
    "publisher", model_client=model_client,
    system_message="최종 승인된 글을 정리하여 발행 형태로 출력하세요.",
)

# 그래프 구성
builder = DiGraphBuilder()
builder.add_node(writer).add_node(reviewer).add_node(publisher)
builder.set_entry_point(writer)

# Writer → Reviewer
builder.add_edge(writer, reviewer)

# 조건 분기: APPROVE 여부에 따라 경로 결정
builder.add_edge(reviewer, publisher,
    condition=lambda msg: "APPROVE" in msg.to_model_text())
builder.add_edge(reviewer, writer,
    condition=lambda msg: "APPROVE" not in msg.to_model_text())

graph = builder.build()

# 실행
termination = MaxMessageTermination(10)
flow = GraphFlow(
    participants=[writer, reviewer, publisher],
    graph=graph,
    termination_condition=termination,
)

await Console(flow.run_stream(task="AI 에이전트의 미래에 대해 글을 작성해주세요."))

### 7-2. 다관점 분석 (병렬 실행)

하나의 리서치 결과를 **3명이 동시에** 분석한 뒤 종합합니다. 병렬 실행은 GraphFlow만의 고유 기능입니다.

```
              → Analyst(긍정) ─┐
Researcher ──┤                 ├→ Synthesizer
              → Analyst(부정) ─┘
```

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import GraphFlow, DiGraphBuilder
from autogen_agentchat.ui import Console

researcher = AssistantAgent(
    "researcher", model_client=model_client,
    system_message="주어진 주제에 대해 핵심 데이터와 사실을 조사하여 보고하세요.",
)
analyst_pro = AssistantAgent(
    "analyst_pro", model_client=model_client,
    system_message="긍정적 관점에서 기회와 장점을 분석하세요.",
)
analyst_con = AssistantAgent(
    "analyst_con", model_client=model_client,
    system_message="비판적 관점에서 리스크와 단점을 분석하세요.",
)
synthesizer = AssistantAgent(
    "synthesizer", model_client=model_client,
    system_message="모든 분석 결과를 종합하여 균형 잡힌 최종 보고서를 작성하세요.",
)

# 그래프 구성
builder = DiGraphBuilder()
builder.add_node(researcher)
builder.add_node(analyst_pro).add_node(analyst_con)
builder.add_node(synthesizer)
builder.set_entry_point(researcher)

# Fan-out: Researcher → 긍정/부정 분석 동시 실행
builder.add_edge(researcher, analyst_pro)
builder.add_edge(researcher, analyst_con)

# Join: 두 분석이 모두 끝나면 → Synthesizer
builder.add_edge(analyst_pro, synthesizer)
builder.add_edge(analyst_con, synthesizer)

graph = builder.build()
flow = GraphFlow(participants=builder.get_participants(), graph=graph)

await Console(flow.run_stream(task="생성형 AI의 사회적 영향을 분석해주세요."))

### 7-3. 고객 문의 라우팅 (조건 분기)

Classifier가 문의를 분류 → 분류 결과에 따라 전문 에이전트로 자동 라우팅

```
              → [BILLING]   → BillingAgent
Classifier ──→ [TECHNICAL] → TechAgent
              → [GENERAL]  → GeneralAgent
```

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import GraphFlow, DiGraphBuilder
from autogen_agentchat.ui import Console

classifier = AssistantAgent(
    "classifier", model_client=model_client,
    system_message="""고객 문의를 분류하세요. 반드시 다음 중 하나를 첫 줄에 출력하세요:
    [BILLING] - 결제/환불 관련
    [TECHNICAL] - 기술 지원 관련
    [GENERAL] - 일반 문의""",
)
billing_agent = AssistantAgent(
    "billing_agent", model_client=model_client,
    system_message="결제/환불 전문 상담원입니다. 고객의 결제 문의를 처리하세요.",
)
tech_agent = AssistantAgent(
    "tech_agent", model_client=model_client,
    system_message="기술 지원 전문 상담원입니다. 고객의 기술 문의를 해결하세요.",
)
general_agent = AssistantAgent(
    "general_agent", model_client=model_client,
    system_message="일반 상담원입니다. 고객의 일반 문의에 응답하세요.",
)

# 그래프 구성
builder = DiGraphBuilder()
builder.add_node(classifier)
builder.add_node(billing_agent).add_node(tech_agent).add_node(general_agent)
builder.set_entry_point(classifier)

builder.add_edge(classifier, billing_agent,
    condition=lambda msg: "[BILLING]" in msg.to_model_text())
builder.add_edge(classifier, tech_agent,
    condition=lambda msg: "[TECHNICAL]" in msg.to_model_text())
builder.add_edge(classifier, general_agent,
    condition=lambda msg: "[GENERAL]" in msg.to_model_text())

graph = builder.build()
flow = GraphFlow(participants=builder.get_participants(), graph=graph)

In [ ]:
# 테스트 1: BILLING → billing_agent로 라우팅
await Console(flow.run_stream(task="지난달 결제한 금액이 이중으로 청구된 것 같습니다. 확인 부탁드립니다."))

In [ ]:
# 테스트 2: TECHNICAL → tech_agent로 라우팅
await Console(flow.run_stream(task="앱이 자꾸 충돌합니다. 최신 버전인데도 로그인 화면에서 멈춰요."))

In [ ]:
# 테스트 3: GENERAL → general_agent로 라우팅
await Console(flow.run_stream(task="영업시간이 어떻게 되나요?"))